# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wizard-i/FlyRank-Starter-Repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### 1. Unit of Analysis

The raw daily fact table contains one row per report date, client, and content page.

For my lane, the final analysis unit is one pseudonymized content page for one client, aggregated across the March 2026 feature window.

### 2. Tables I Will Use

- `fact_content_daily_performance`: March search-performance features and April outcome measurements.
- `dim_content`: content creation date, used to calculate content age at the decision cutoff.

I will not use the query-level table in this first contract.

### 3. Time Windows

- Feature window: March 1–31, 2026.
- Decision cutoff: March 31, 2026.
- Outcome window: April 1–30, 2026.

All five features must be knowable by the March 31 decision cutoff. April measurements are reserved for the outcome label.

### 4. Target / Proxy

My outcome proxy is `went_dark_apr`.

It equals 1 when a page has zero Google Search Console clicks across at least 20 observed GSC-available days in April. It equals 0 when at least one click is observed.

This measures a future no-click outcome, not whether a content refresh will cause recovery.

### 5. One Deliberate Exclusion

I deliberately exclude the entire `fact_content_query_90d` table because its fixed 90-day window does not align safely with my March feature and April outcome windows.

### Expected Human Output

A ranked review queue that helps content editors decide which pages deserve further investigation and possible refresh actions.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



%pip -q install -U duckdb

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

# Read the token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add it in Colab Secrets first."
    )

# Connect DuckDB securely
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Keep the secret out of the SQL source text
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])

con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

# Warehouse paths
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

# Mid-panel feature and outcome windows
MARCH = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{FACT}/month=2026-04/*.parquet')"

# Supporting content dimension
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print("DuckDB connected successfully.")
print("Feature window: March 2026")
print("Outcome window: April 2026")


DuckDB connected successfully.
Feature window: March 2026
Outcome window: April 2026


## 2. Fields: feature / label / context / excluded


### Five Features

| Feature | Meaning | Knowable at the decision moment because... |
|---|---|---|
| `impressions_mar` | Total GSC impressions in March | March search exposure was measured before the March 31 cutoff. |
| `clicks_mar` | Total GSC clicks in March | These clicks were already observed during the feature window. |
| `ctr_mar_pct` | March clicks divided by March impressions | Both measurements were available by March 31. |
| `avg_position_mar` | Impression-weighted average search position | It is calculated only from March position measurements. |
| `content_age_days_mar` | Age of the page on March 31 | It is calculated from the page's creation date and the fixed cutoff date. |

### Label / Proxy

- `went_dark_apr`: Zero observed GSC clicks across at least 20 GSC-available days in April.

### Context

- `client_hash_id`: Identifies the client for grouping and joining.
- `content_hash_id`: Identifies the content page for grouping and joining.
- `report_date`: Identifies the observation date.
- `gsc_data_available`: Identifies whether GSC data is measured.
- `gsc_days_apr`: Tracks the amount of measured April data for outcome eligibility.

These fields are not model features.

### Excluded

- `fact_content_query_90d`: excluded because its fixed 90-day measurement window overlaps the selected time windows.
- `last_optimized_date` and `has_been_optimized`: excluded because these can reflect product decisions or information unavailable at the March 31 cutoff.
- Client and content identifiers: context only, never predictive features.

### Missing Data

A row with `gsc_data_available = FALSE` is not evidence of zero search traffic.

I will only use measured GSC rows for the selected features and outcome. I will not treat unavailable GA4 measurements as zero engagement.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# BUILD APRIL OUTCOME LABEL

label_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    CAST(SUM(gsc_sum_position) AS DOUBLE)
        / NULLIF(SUM(gsc_impressions), 0)
        AS avg_gsc_position_apr,

    SUM(gsc_impressions)
        AS total_gsc_impressions_apr,

    COUNT(DISTINCT report_date)
        AS n_gsc_days_apr

FROM read_parquet('{APRIL_PATH}')

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id

HAVING SUM(gsc_impressions) >= 100
"""

april_outcomes = con.sql(label_query).df()

print("APRIL OUTCOME DATA")
print("=" * 40)

print("April eligible page-client pairs:", len(april_outcomes))

display(april_outcomes.head(5))


NameError: name 'APRIL_PATH' is not defined

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:

# QUERY 1: Verify the raw daily grain

grain_query = con.sql(f"""
    SELECT COUNT(*) AS duplicate_grain_groups
    FROM (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id
        FROM {MARCH}
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    ) AS duplicates
""").df()

display(grain_query)



In [ ]:

# QUERY 2: Verify row count and date span

count_query = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date) AS distinct_days,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_content_items
    FROM {MARCH}
""").df()

display(count_query)


In [ ]:

# QUERY 3: Verify measured data availability

availability_query = con.sql(f"""
    SELECT
        'GSC available' AS data_type,
        COUNT(*) AS surviving_rows
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE

    UNION ALL

    SELECT
        'GA4 available',
        COUNT(*)
    FROM {MARCH}
    WHERE ga4_data_available IS TRUE

    UNION ALL

    SELECT
        'Both available',
        COUNT(*)
    FROM {MARCH}
    WHERE
        gsc_data_available IS TRUE
        AND ga4_data_available IS TRUE
""").df()

display(availability_query)



### Five Features and Their Availability

My feature frame contains only five predictive features.

1. `impressions_mar`: Knowable by March 31 because it aggregates measured March search impressions.
2. `clicks_mar`: Knowable by March 31 because it aggregates measured March search clicks.
3. `ctr_mar_pct`: Knowable by March 31 because it is calculated from March clicks and impressions.
4. `avg_position_mar`: Knowable by March 31 because it is calculated from March impression-weighted position measurements.
5. `content_age_days_mar`: Knowable by March 31 because it uses the content creation date and a fixed March 31 cutoff.

The five features are based on historical March observations. April measurements are reserved exclusively for the outcome label.


In [ ]:


# Build one row per client-content pair using March data

march_features = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(f.gsc_impressions) AS impressions_mar,

        SUM(f.gsc_clicks) AS clicks_mar,

        100.0 * SUM(f.gsc_clicks)
            / NULLIF(SUM(f.gsc_impressions), 0)
            AS ctr_mar_pct,

        SUM(f.gsc_sum_position)
            / NULLIF(SUM(f.gsc_impressions), 0)
            AS avg_position_mar,

        DATE_DIFF(
            'day',
            CAST(d.content_created_date AS DATE),
            DATE '2026-03-31'
        ) AS content_age_days_mar

    FROM {MARCH} AS f

    INNER JOIN {DIM_CONTENT} AS d
        ON f.content_hash_id = d.content_hash_id

    WHERE f.gsc_data_available IS TRUE
      AND d.content_created_date IS NOT NULL
      AND CAST(d.content_created_date AS DATE)
          <= DATE '2026-03-31'

    GROUP BY
        f.client_hash_id,
        f.content_hash_id,
        d.content_created_date

    HAVING SUM(f.gsc_impressions) >= 100
       AND SUM(f.gsc_clicks) >= 3

""").df()

FEATURE_COLUMNS = [
    "impressions_mar",
    "clicks_mar",
    "ctr_mar_pct",
    "avg_position_mar",
    "content_age_days_mar"
]

print("March feature frame created.")
print("Number of page-client rows:", len(march_features))
print("Number of predictive features:", len(FEATURE_COLUMNS))

display(
    march_features[FEATURE_COLUMNS].head(10)
)


In [ ]:

# Build April outcome using measured GSC data only

april_outcomes = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_clicks) AS clicks_apr,

        COUNT(DISTINCT report_date) AS gsc_days_apr

    FROM {APRIL}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING COUNT(DISTINCT report_date) >= 20

""").df()

print("April outcome data prepared.")
print("Eligible page-client pairs:", len(april_outcomes))


In [ ]:

# Match March features with April outcomes

feature_frame = march_features.merge(
    april_outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    validate="one_to_one"
)

# Create the future outcome proxy
feature_frame["went_dark_apr"] = (
    feature_frame["clicks_apr"] == 0
).astype("int8")

print("FINAL FEATURE FRAME")
print("=" * 40)

print("Matched page-client rows:", len(feature_frame))

print("\nTarget distribution:")
print(
    feature_frame["went_dark_apr"]
    .value_counts()
    .sort_index()
)

print("\nTarget percentage:")
print(
    (
        feature_frame["went_dark_apr"]
        .value_counts(normalize=True)
        * 100
    ).round(2)
)

# Show only safe analytical columns, not identifiers
display(
    feature_frame[
        FEATURE_COLUMNS + ["went_dark_apr"]
    ].head(10)
)


In [ ]:

# Deliberate leakage experiment
# IMPORTANT: This is a demonstration, not a valid model result.

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

y = feature_frame["went_dark_apr"].astype(int)

if y.nunique() != 2:
    raise ValueError(
        "Leakage experiment requires both target classes."
    )

# Honest model: five March features only
X_honest = feature_frame[FEATURE_COLUMNS].copy()

# Keep the exact same train/test indices for both experiments
train_idx, test_idx = train_test_split(
    np.arange(len(feature_frame)),
    test_size=0.25,
    random_state=42,
    stratify=y
)

def quick_auc(X):

    model = make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeClassifier(
            max_depth=3,
            random_state=42
        )
    )

    model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    probabilities = model.predict_proba(
        X.iloc[test_idx]
    )[:, 1]

    return roc_auc_score(
        y.iloc[test_idx],
        probabilities
    )

# Honest score
honest_auc = quick_auc(X_honest)

# Deliberately add April's label-derived source
X_leaky = X_honest.copy()

X_leaky["april_clicks_LEAK"] = (
    feature_frame["clicks_apr"].astype(float)
)

# Leaked score
leaky_auc = quick_auc(X_leaky)

print("DATA LEAKAGE EXPERIMENT")
print("=" * 40)

print(f"Honest ROC-AUC (5 features): {honest_auc:.4f}")
print(f"Leaked ROC-AUC (with April clicks): {leaky_auc:.4f}")

print("\nThe difference demonstrates how future information")
print("can artificially inflate model performance.")

# Remove the deliberately leaked feature
del X_leaky

# Remove the April click source from the final frame
feature_frame = feature_frame.drop(
    columns=["clicks_apr"]
)

X_final = feature_frame[FEATURE_COLUMNS].copy()

assert "april_clicks_LEAK" not in X_final.columns
assert len(FEATURE_COLUMNS) == 5

print("\nLeakage feature removed successfully.")
print("Final feature count:", len(X_final.columns))
print("Honest ROC-AUC to retain:", round(honest_auc, 4))
print("Final feature columns:", list(X_final.columns))




## 4. Data Limits

### Limitation: Unbalanced Panel and Incomplete Future Coverage

The warehouse is an unbalanced panel. Different clients have different amounts of historical data, and GSC availability is not identical across all page-client pairs.

For this initial experiment, I require at least 20 GSC-available days in April before assigning an outcome label.

This means pages with insufficient April observations are excluded from the labelled cohort. The resulting sample may therefore favour pages and clients with more consistent measurement coverage.

Another limitation is that zero observed clicks do not prove that a page needs refreshing or that refreshing it will restore performance. Seasonal changes, demand shifts, consolidation, and measurement gaps may also explain the observed outcome.

The result should be used as decision-support evidence, not as a guarantee of content recovery.


In [ ]:

# Report the effect of the April coverage requirement

print("DATA COVERAGE LIMITATION SUMMARY")
print("=" * 40)

print(
    "March eligible page-client pairs:",
    len(march_features)
)

print(
    "April pairs with at least 20 measured days:",
    len(april_outcomes)
)

print(
    "Matched and labelled pairs:",
    len(feature_frame)
)

print(
    "March pairs without an eligible April outcome:",
    len(march_features) - len(feature_frame)
)

print(
    "\nFinal predictive feature count:",
    len(FEATURE_COLUMNS)
)



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.